# **API data collection**

In [ ]:
import requests
import pandas as pd
url = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL"

params = {
    "date": "2023",
    "format": "json",
    "per_page": 500
}
response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
data = response.json()
print("HTTP status:", response.status_code)
print("Number of records:", len(data[1]))
print("First record:", data[1][0])

HTTP status: 200
Number of records: 266
First record: {'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'}, 'country': {'id': 'ZH', 'value': 'Africa Eastern and Southern'}, 'countryiso3code': 'AFE', 'date': '2023', 'value': 750491370, 'unit': '', 'obs_status': '', 'decimal': 0}


In [ ]:
df_population = pd.DataFrame(data[1])

df_population = pd.DataFrame({
    "Country": df_population["country"].apply(lambda x: x["value"]),
    "Country_Code": df_population["countryiso3code"],
    "Population": df_population["value"]
})

df_population.head()

,Country,Country_Code,Population
0,Africa Eastern and Southern,AFE,750491370.0
1,Africa Western and Central,AFW,508318102.0
2,Arab World,ARB,482105978.0
3,Caribbean small states,CSS,4519904.0
4,Central Europe and the Baltics,CEB,100183707.0


In [ ]:
gdp_url = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.PCAP.CD"

gdp_response = requests.get(gdp_url, params=params, timeout=30)
gdp_response.raise_for_status()

gdp_data = gdp_response.json()

df_gdp = pd.DataFrame(gdp_data[1])

df_gdp = pd.DataFrame({
    "Country_Code": df_gdp["countryiso3code"],
    "GDP_per_capita": df_gdp["value"]
})
df_gdp.head()

,Country_Code,GDP_per_capita
0,AFE,1571.132704
1,AFW,1846.246811
2,ARB,7504.356377
3,CSS,17609.682707
4,CEB,22693.950447


In [ ]:
life_url = "https://api.worldbank.org/v2/country/all/indicator/SP.DYN.LE00.IN"

life_response = requests.get(life_url, params=params, timeout=30)
life_response.raise_for_status()

life_data = life_response.json()

df_life = pd.DataFrame(life_data[1])

df_life = pd.DataFrame({
    "Country_Code": df_life["countryiso3code"],
    "Life_expectancy": df_life["value"]
})

df_life.head()

,Country_Code,Life_expectancy
0,AFE,65.146228
1,AFW,58.847483
2,ARB,72.444678
3,CSS,73.171645
4,CEB,77.758038


In [ ]:
df_api = df_population.merge(
    df_gdp,
    on="Country_Code",
    how="left"
)

df_api = df_api.merge(
    df_life,
    on="Country_Code",
    how="left"
)
df_api.head()

,Country,Country_Code,Population,GDP_per_capita,Life_expectancy
0,Africa Eastern and Southern,AFE,750491370.0,1571.132704,65.146228
1,Africa Western and Central,AFW,508318102.0,1846.246811,58.847483
2,Arab World,ARB,482105978.0,7504.356377,72.444678
3,Caribbean small states,CSS,4519904.0,17609.682707,73.171645
4,Central Europe and the Baltics,CEB,100183707.0,22693.950447,77.758038


In [ ]:
countries_url = "https://api.worldbank.org/v2/country"

countries_params = {
    "format": "json",
    "per_page": 500
}
countries_response = requests.get(
    countries_url,
    params=countries_params,
    timeout=30
)
countries_response.raise_for_status()

countries_data = countries_response.json()[1]
df_countries = pd.DataFrame(countries_data)

valid_codes = df_countries.loc[
    df_countries["region"].apply(
        lambda x: x["value"] != "Aggregates"
    ),
    "id"
]
df_api = df_api[
    df_api["Country_Code"].isin(valid_codes)
].copy()

df_api = df_api.reset_index(drop=True)

print("Number of countries:", len(df_api))
df_api.head()

Number of countries: 217


,Country,Country_Code,Population,GDP_per_capita,Life_expectancy
0,Afghanistan,AFG,41454761.0,413.757895,66.035
1,Albania,ALB,2411658.0,9740.702341,79.602
2,Algeria,DZA,46164219.0,5370.477235,76.261
3,American Samoa,ASM,47521.0,NaN,72.852
4,Andorra,AND,80856.0,46812.448450,84.041


In [ ]:
print("Dataset shape:", df_api.shape)
print("\nMissing values:")
print(df_api.isnull().sum())
print("\nDuplicates:", df_api.duplicated().sum())
print("\nData types:")
print(df_api.dtypes)

Dataset shape: (217, 5)

Missing values:
Country             0
Country_Code        0
Population          0
GDP_per_capita     13
Life_expectancy     0
dtype: int64

Duplicates: 0

Data types:
Country             object
Country_Code        object
Population         float64
GDP_per_capita     float64
Life_expectancy    float64
dtype: object


In [ ]:
df_api = df_api.dropna(
    subset=["GDP_per_capita"]
).copy()

df_api = df_api.drop_duplicates(
    subset=["Country_Code"]
)
df_api = df_api.reset_index(drop=True)
print("Dataset after cleaning:", df_api.shape)
print("\nMissing values:")
print(df_api.isnull().sum())
df_api.head()

Dataset after cleaning: (204, 5)

Missing values:
Country            0
Country_Code       0
Population         0
GDP_per_capita     0
Life_expectancy    0
dtype: int64


,Country,Country_Code,Population,GDP_per_capita,Life_expectancy
0,Afghanistan,AFG,41454761.0,413.757895,66.035
1,Albania,ALB,2411658.0,9740.702341,79.602
2,Algeria,DZA,46164219.0,5370.477235,76.261
3,Andorra,AND,80856.0,46812.448450,84.041
4,Angola,AGO,36749906.0,2885.513491,64.617


In [ ]:
df_api.to_csv("world_bank_cleaned_2023.csv", index=False)
print("CSV file saved successfully!")
print("Final dataset shape:", df_api.shape)
df_api.sample(5, random_state=42)

CSV file saved successfully!
Final dataset shape: (204, 5)


,Country,Country_Code,Population,GDP_per_capita,Life_expectancy
15,Barbados,BRB,282336.0,25583.872230,76.17800
9,Australia,AUS,26659922.0,65058.377315,83.05122
115,Mali,MLI,23769127.0,1042.442234,60.43900
78,Haiti,HTI,11637398.0,1681.844077,64.93600
66,Gabon,GAB,2484789.0,7802.824333,68.33700


# **Web Scrapping**

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

wiki_url = "https://en.wikipedia.org/wiki/World_Happiness_Report"
headers = {"User-Agent": "KBTU-DataCollection-Project/1.0 (student project)"}

robots_response = requests.get("https://en.wikipedia.org/robots.txt", headers=headers, timeout=30)
robots_parser = RobotFileParser()
robots_parser.parse(robots_response.text.splitlines())

print("robots.txt status:", robots_response.status_code)
print("Scraping allowed:", robots_parser.can_fetch(headers["User-Agent"], wiki_url))

robots.txt status: 200
Scraping allowed: True


In [ ]:
wiki_response = requests.get(wiki_url, headers=headers, timeout=30)
wiki_response.raise_for_status()

soup = BeautifulSoup(wiki_response.text, "html.parser")

print("HTTP status:", wiki_response.status_code)
print("Page size (characters):", len(wiki_response.text))
print("Page title:", soup.title.get_text())

HTTP status: 200
Page size (characters): 3404900
Page title: World Happiness Report - Wikipedia


In [ ]:
tables = soup.find_all("table", class_="wikitable")
print("Number of wikitables on the page:", len(tables))

for i, table in enumerate(tables):
    first_row = table.find("tr")
    if first_row is None:
        continue
    header = [th.get_text(strip=True) for th in first_row.find_all("th")]
    print(i, header[:5])

Number of wikitables on the page: 13
0 ['Rank', 'Country or region', 'Life evaluation', 'GDP per capita', 'Social support']
1 ['Rank', 'Country or region', 'Life evaluation', 'GDP per capita', 'Social support']
2 ['Rank', 'Country or region', 'Life evaluation']
3 ['Rank', 'Country or region']
4 ['Rank', 'Country or region']
5 ['Rank', 'Country or region', 'Life evaluation', 'GDP per capita', 'Social support']
6 ['Rank', 'Country or region', 'Life evaluation', 'GDP per capita', 'Social support']
7 ['Rank', 'Country or region', 'Life evaluation', 'GDP per capita', 'Social support']
8 ['Rank', 'Change in rank', 'Country', 'Life evaluation', 'Change in score']
9 ['Rank', 'Country', 'Life evaluation', 'Change fromprior year', 'GDP per capita']
10 ['Rank[30]', 'Country', 'Life evaluation']
11 ['Overall rank', 'Country or region', 'Life evaluation']
12 ['Overall rank', 'Country or region', 'Life evaluation']


In [ ]:
report_heading = soup.find(id="2024_report")

if report_heading is None:
    raise ValueError("Section '2024 report' not found")

happiness_table = report_heading.find_next("table", class_="wikitable")

header = [th.get_text(strip=True) for th in happiness_table.find("tr").find_all("th")]
print("Section:", report_heading.get_text(strip=True))
print("Table columns:", header)
print("Number of rows:", len(happiness_table.find_all("tr")) - 1)

Section: 2024 report
Table columns: ['Rank', 'Country or region', 'Life evaluation']
Number of rows: 143


In [ ]:
rows = []

for tr in happiness_table.find_all("tr")[1:]:
    cells = [cell.get_text(strip=True) for cell in tr.find_all(["td", "th"])]
    if len(cells) >= 3:
        rows.append({
            "Rank": cells[0],
            "Country": cells[1],
            "Happiness": cells[2]
        })

df_happiness = pd.DataFrame(rows)
df_happiness.head(10)

,Rank,Country,Happiness
0,1,Finland,7.741
1,2,Denmark,7.583
2,3,Iceland,7.525
3,4,Sweden,7.344
4,5,Israel,7.341
5,6,Netherlands,7.319
6,7,Norway,7.302
7,8,Luxembourg,7.122
8,9,Switzerland,7.060
9,10,Australia,7.057


In [ ]:
print("Dataset shape:", df_happiness.shape)
print("\nMissing values:")
print(df_happiness.isnull().sum())
print("\nDuplicates:", df_happiness.duplicated(subset=["Country"]).sum())
print("\nData types:")
print(df_happiness.dtypes)

Dataset shape: (143, 3)

Missing values:
Rank         0
Country      0
Happiness    0
dtype: int64

Duplicates: 0

Data types:
Rank         object
Country      object
Happiness    object
dtype: object


In [ ]:
df_happiness["Country"] = (
    df_happiness["Country"]
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.replace("*", "", regex=False)
    .str.strip()
)
df_happiness["Happiness"] = pd.to_numeric(df_happiness["Happiness"], errors="coerce")
df_happiness["Rank"] = pd.to_numeric(df_happiness["Rank"], errors="coerce").astype("Int64")

df_happiness = df_happiness.dropna(subset=["Happiness"])
df_happiness = df_happiness.drop_duplicates(subset=["Country"])
df_happiness = df_happiness.reset_index(drop=True)

print("Dataset after cleaning:", df_happiness.shape)
print("\nData types:")
print(df_happiness.dtypes)

Dataset after cleaning: (143, 3)

Data types:
Rank           Int64
Country       object
Happiness    float64
dtype: object


In [ ]:
print("Score range:", df_happiness["Happiness"].min(), "-", df_happiness["Happiness"].max())
print("All scores between 0 and 10:", df_happiness["Happiness"].between(0, 10).all())

df_happiness["Happiness_Level"] = pd.cut(
    df_happiness["Happiness"],
    bins=[0, 4.5, 6, 10],
    labels=["Low", "Medium", "High"]
)

print("\nCountries by happiness level:")
print(df_happiness["Happiness_Level"].value_counts())

print("\nTop 5 happiest countries:")
print(df_happiness.head(5).to_string(index=False))

print("\nTop 5 least happy countries:")
print(df_happiness.tail(5).to_string(index=False))

Score range: 1.721 - 7.741
All scores between 0 and 10: True

Countries by happiness level:
Happiness_Level
High      56
Medium    55
Low       32
Name: count, dtype: int64

Top 5 happiest countries:
 Rank Country  Happiness Happiness_Level
    1 Finland      7.741            High
    2 Denmark      7.583            High
    3 Iceland      7.525            High
    4  Sweden      7.344            High
    5  Israel      7.341            High

Top 5 least happy countries:
 Rank          Country  Happiness Happiness_Level
  139 Congo (Kinshasa)      3.295             Low
  140     Sierra Leone      3.245             Low
  141          Lesotho      3.186             Low
  142          Lebanon      2.707             Low
  143      Afghanistan      1.721             Low


In [ ]:
country_name_map = {
    "Russia": "Russian Federation",
    "Turkey": "Turkiye",
    "South Korea": "Korea, Rep.",
    "Iran": "Iran, Islamic Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "Venezuela": "Venezuela, RB",
    "Laos": "Lao PDR",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Slovakia": "Slovak Republic",
    "Vietnam": "Viet Nam",
    "Yemen": "Yemen, Rep.",
    "Gambia": "Gambia, The",
    "Ivory Coast": "Cote d'Ivoire",
    "Hong Kong": "Hong Kong SAR, China",
    "Congo (Brazzaville)": "Congo, Rep.",
    "Congo (Kinshasa)": "Congo, Dem. Rep.",
    "State of Palestine": "West Bank and Gaza"
}

df_happiness["Country"] = df_happiness["Country"].replace(country_name_map)

if "df_api" in globals():
    unmatched = df_happiness.loc[~df_happiness["Country"].isin(df_api["Country"]), "Country"]
    print("Countries not found in World Bank data:", len(unmatched))
    print(unmatched.tolist())

In [ ]:
df_happiness.to_csv("happiness_2024_cleaned.csv", index=False)

print("CSV file saved successfully!")
print("Final dataset shape:", df_happiness.shape)
df_happiness.sample(5, random_state=42)

CSV file saved successfully!
Final dataset shape: (143, 4)


,Rank,Country,Happiness,Happiness_Level
117,118,Myanmar,4.354,Low
19,20,United Kingdom,6.749,High
82,83,South Africa,5.422,Medium
97,98,Turkiye,4.975,Medium
56,57,Paraguay,5.977,Medium


# **Data Analysis**